
# 02 — Dataset Validation

## Objective
Validate the University of Nottingham examination timetabling
benchmark before constructing the examination conflict graph.

This notebook checks:
- Dataset file availability and record counts
- Unique student and examination identifiers
- Enrolment data integrity
- Missing or invalid references
- Duplicate student–exam enrolment pairs
- Basic enrolment statistics

The validated data will be used in the subsequent
conflict graph construction and algorithm evaluation stages.


In [1]:

from pathlib import Path
import pandas as pd
from collections import Counter

# Locate the project root from this notebook's working directory
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data" / "raw" / "Nott").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate data/raw/Nott. "
            "Open this notebook inside the project."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Nott"

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)
print("Dataset directory exists:", DATA_DIR.exists())


Project root: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling
Dataset directory: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\raw\Nott
Dataset directory exists: True


In [2]:

# Expected benchmark files
expected_files = ["students", "exams", "enrolements"]

file_report = []

for filename in expected_files:
    file_path = DATA_DIR / filename

    if file_path.exists():
        with open(file_path, "r", encoding="utf-8", errors="replace") as f:
            line_count = sum(1 for _ in f)

        file_report.append({
            "File": filename,
            "Exists": True,
            "Records": line_count,
            "Size (KB)": round(file_path.stat().st_size / 1024, 2)
        })
    else:
        file_report.append({
            "File": filename,
            "Exists": False,
            "Records": None,
            "Size (KB)": None
        })

file_report_df = pd.DataFrame(file_report)
display(file_report_df)

assert all(file_report_df["Exists"]), (
    "One or more required benchmark files are missing."
)

print("All required dataset files are available.")


,File,Exists,Records,Size (KB)
0,students,True,7896,123.38
1,exams,True,800,45.31
2,enrolements,True,33997,664.00


All required dataset files are available.


In [3]:

# Read student IDs (10 characters per record)
with open(DATA_DIR / "students", "r", encoding="utf-8", errors="replace") as f:
    student_ids = [
        line[:10].strip()
        for line in f
        if line.strip()
    ]

# Read exam IDs (first 8 characters of each fixed-width record)
with open(DATA_DIR / "exams", "r", encoding="utf-8", errors="replace") as f:
    exam_ids = [
        line[:8].strip()
        for line in f
        if line.strip()
    ]

# Read enrolments: student ID (10 characters) + exam ID (8 characters)
student_exam_pairs = []

with open(DATA_DIR / "enrolements", "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.rstrip("\r\n")

        if len(line) >= 18:
            student_id = line[:10].strip()
            exam_id = line[10:18].strip()

            if student_id and exam_id:
                student_exam_pairs.append((student_id, exam_id))

# Build lookup sets
student_id_set = set(student_ids)
exam_id_set = set(exam_ids)

# Check duplicate identifiers and enrolment pairs
duplicate_student_ids = len(student_ids) - len(student_id_set)
duplicate_exam_ids = len(exam_ids) - len(exam_id_set)
duplicate_enrolments = (
    len(student_exam_pairs) - len(set(student_exam_pairs))
)

# Check enrolment references
unknown_students = {
    student_id
    for student_id, exam_id in student_exam_pairs
    if student_id not in student_id_set
}

unknown_exams = {
    exam_id
    for student_id, exam_id in student_exam_pairs
    if exam_id not in exam_id_set
}

validation_summary = pd.DataFrame([
    {"Check": "Student records", "Result": len(student_ids)},
    {"Check": "Unique student IDs", "Result": len(student_id_set)},
    {"Check": "Duplicate student IDs", "Result": duplicate_student_ids},
    {"Check": "Exam records", "Result": len(exam_ids)},
    {"Check": "Unique exam IDs", "Result": len(exam_id_set)},
    {"Check": "Duplicate exam IDs", "Result": duplicate_exam_ids},
    {"Check": "Enrolment records", "Result": len(student_exam_pairs)},
    {"Check": "Duplicate student-exam pairs", "Result": duplicate_enrolments},
    {"Check": "Unknown student references", "Result": len(unknown_students)},
    {"Check": "Unknown exam references", "Result": len(unknown_exams)},
])

display(validation_summary)

print("Identifier and enrolment validation completed.")


,Check,Result
0,Student records,7896
1,Unique student IDs,7896
2,Duplicate student IDs,0
3,Exam records,800
4,Unique exam IDs,800
5,Duplicate exam IDs,0
6,Enrolment records,33997
7,Duplicate student-exam pairs,131
8,Unknown student references,0
9,Unknown exam references,791


Identifier and enrolment validation completed.


In [4]:

from collections import Counter

# Count unknown exam IDs and identify the most common ones
unknown_exam_counts = Counter(
    exam_id
    for student_id, exam_id in student_exam_pairs
    if exam_id not in exam_id_set
)

print("Number of distinct unknown exam IDs:", len(unknown_exam_counts))
print("Total enrolment records with unknown exam IDs:", sum(unknown_exam_counts.values()))

print("\nFirst 20 unknown exam IDs:")
for exam_id, count in unknown_exam_counts.most_common(20):
    print(f"{exam_id!r}: {count} enrolment records")

print("\nFirst 10 exam IDs from the exam file:")
print(sorted(exam_id_set)[:10])

print("\nFirst 10 exam IDs from enrolments:")
print(sorted({exam_id for _, exam_id in student_exam_pairs})[:10])


Number of distinct unknown exam IDs: 791
Total enrolment records with unknown exam IDs: 33997

First 20 unknown exam IDs:
'HGAEM2E': 542 enrolment records
'HGBEM3E': 340 enrolment records
'G111GBE': 222 enrolment records
'C71101E': 205 enrolment records
'R11001E': 191 enrolment records
'M3A010E': 187 enrolment records
'M3B030E': 184 enrolment records
'M3A015E': 176 enrolment records
'F311T1E': 175 enrolment records
'G11MA1E': 172 enrolment records
'M31001E': 171 enrolment records
'M31012E': 168 enrolment records
'G111VAE': 165 enrolment records
'G11PR1E': 165 enrolment records
'C11105E': 163 enrolment records
'F311M1E': 161 enrolment records
'L11100E': 160 enrolment records
'F311Q1E': 157 enrolment records
'M3A002E': 155 enrolment records
'N11102E': 155 enrolment records

First 10 exam IDs from the exam file:
['AA2016E1', 'AA3008E1', 'AAA011E1', 'AAA013E1', 'AAA022E1', 'AAA024E1', 'AAA033E1', 'AAA034E1', 'AAB025E1', 'AAB058E1']

First 10 exam IDs from enrolments:
['AA2016E', 'AA3008E',

In [5]:

# Normalize enrolment exam IDs using the authoritative exam ID set.
# Append "1" only when the resulting ID exists in the exam file.

normalized_pairs = []
unmatched_exam_ids = set()

for student_id, original_exam_id in student_exam_pairs:
    if original_exam_id in exam_id_set:
        normalized_exam_id = original_exam_id
    elif original_exam_id + "1" in exam_id_set:
        normalized_exam_id = original_exam_id + "1"
    else:
        unmatched_exam_ids.add(original_exam_id)
        continue

    normalized_pairs.append((student_id, normalized_exam_id))

normalized_pair_counts = Counter(normalized_pairs)
duplicate_normalized_pairs = sum(
    count - 1
    for count in normalized_pair_counts.values()
    if count > 1
)

normalized_unknown_students = {
    student_id
    for student_id, exam_id in normalized_pairs
    if student_id not in student_id_set
}

normalized_unknown_exams = {
    exam_id
    for student_id, exam_id in normalized_pairs
    if exam_id not in exam_id_set
}

print("Original enrolment records:", len(student_exam_pairs))
print("Successfully normalized enrolments:", len(normalized_pairs))
print("Unmatched original exam IDs:", len(unmatched_exam_ids))
print("Duplicate pairs after normalization:", duplicate_normalized_pairs)
print("Unknown student references after normalization:", len(normalized_unknown_students))
print("Unknown exam references after normalization:", len(normalized_unknown_exams))

if unmatched_exam_ids:
    print("\nUnmatched IDs (first 20):")
    print(sorted(unmatched_exam_ids)[:20])


Original enrolment records: 33997
Successfully normalized enrolments: 33939
Unmatched original exam IDs: 1
Duplicate pairs after normalization: 131
Unknown student references after normalization: 0
Unknown exam references after normalization: 0

Unmatched IDs (first 20):
['K1AHWAE']


In [6]:

# Handle the confirmed special exam ID
special_exam_mapping = {
    "K1AHWAE": "K1AHWAE2"
}

final_pairs = []
unmatched_final_ids = set()

for student_id, original_exam_id in student_exam_pairs:
    if original_exam_id in exam_id_set:
        normalized_exam_id = original_exam_id
    elif original_exam_id in special_exam_mapping:
        normalized_exam_id = special_exam_mapping[original_exam_id]
    elif original_exam_id + "1" in exam_id_set:
        normalized_exam_id = original_exam_id + "1"
    else:
        unmatched_final_ids.add(original_exam_id)
        continue

    final_pairs.append((student_id, normalized_exam_id))

final_pair_counts = Counter(final_pairs)

duplicate_final_pairs = {
    pair: count
    for pair, count in final_pair_counts.items()
    if count > 1
}

final_unknown_students = {
    student_id
    for student_id, exam_id in final_pairs
    if student_id not in student_id_set
}

final_unknown_exams = {
    exam_id
    for student_id, exam_id in final_pairs
    if exam_id not in exam_id_set
}

print("Original enrolment records:", len(student_exam_pairs))
print("Mapped enrolment records:", len(final_pairs))
print("Unmatched exam IDs:", len(unmatched_final_ids))
print("Unknown student references:", len(final_unknown_students))
print("Unknown exam references:", len(final_unknown_exams))
print("Duplicate pairs after normalization:", len(duplicate_final_pairs))
print("Extra occurrences among duplicate pairs:",
      sum(count - 1 for count in duplicate_final_pairs.values()))

print("\nSpecial mapping occurrences:",
      sum(1 for _, exam_id in student_exam_pairs
          if exam_id == "K1AHWAE"))

if duplicate_final_pairs:
    print("\nFirst 10 normalized duplicate pairs:")
    for pair, count in list(duplicate_final_pairs.items())[:10]:
        print(pair, "occurrences:", count)


Original enrolment records: 33997
Mapped enrolment records: 33997
Unmatched exam IDs: 0
Unknown student references: 0
Unknown exam references: 0
Duplicate pairs after normalization: 122
Extra occurrences among duplicate pairs: 131

Special mapping occurrences: 58

First 10 normalized duplicate pairs:
('A940014523', 'C81J5AE1') occurrences: 2
('A940022595', 'C81J3AE1') occurrences: 2
('A940026445', 'C81J2AE1') occurrences: 2
('A940031287', 'C81J2AE1') occurrences: 2
('A940033292', 'C81J2AE1') occurrences: 2
('A940034646', 'C81J2AE1') occurrences: 2
('A940042722', 'C81J3AE1') occurrences: 2
('A945073318', 'C81J4AE1') occurrences: 2
('A945311483', 'C81J3AE1') occurrences: 2
('A946008014', 'C81J5AE1') occurrences: 2


In [7]:

# Examine the raw records behind normalized duplicate pairs

duplicate_examples = []

for (student_id, normalized_exam_id), count in duplicate_final_pairs.items():
    original_ids = sorted({
        original_exam_id
        for original_student_id, original_exam_id in student_exam_pairs
        if original_student_id == student_id
        and (
            original_exam_id == normalized_exam_id
            or original_exam_id + "1" == normalized_exam_id
            or special_exam_mapping.get(original_exam_id) == normalized_exam_id
        )
    })

    duplicate_examples.append({
        "Student ID": student_id,
        "Normalized exam ID": normalized_exam_id,
        "Occurrences": count,
        "Original exam IDs": ", ".join(original_ids)
    })

duplicate_examples_df = pd.DataFrame(duplicate_examples)

display(duplicate_examples_df.head(20))

print("Distinct duplicated normalized pairs:", len(duplicate_examples_df))
print(
    "Pairs with multiple original exam IDs:",
    sum(
        len(item.split(", ")) > 1
        for item in duplicate_examples_df["Original exam IDs"]
    )
)


,Student ID,Normalized exam ID,Occurrences,Original exam IDs
0,A940014523,C81J5AE1,2,C81J5AE
1,A940022595,C81J3AE1,2,C81J3AE
2,A940026445,C81J2AE1,2,C81J2AE
3,A940031287,C81J2AE1,2,C81J2AE
4,A940033292,C81J2AE1,2,C81J2AE
5,A940034646,C81J2AE1,2,C81J2AE
6,A940042722,C81J3AE1,2,C81J3AE
7,A945073318,C81J4AE1,2,C81J4AE
8,A945311483,C81J3AE1,2,C81J3AE
9,A946008014,C81J5AE1,2,C81J5AE


Distinct duplicated normalized pairs: 122
Pairs with multiple original exam IDs: 0


In [8]:

# Count repeated pairs directly in the raw enrolment file
raw_pair_counts = Counter(student_exam_pairs)

raw_duplicate_pairs = {
    pair: count
    for pair, count in raw_pair_counts.items()
    if count > 1
}

print("Raw enrolment records:", len(student_exam_pairs))
print("Unique raw student-exam pairs:", len(raw_pair_counts))
print("Distinct repeated raw pairs:", len(raw_duplicate_pairs))
print(
    "Extra raw occurrences:",
    sum(count - 1 for count in raw_duplicate_pairs.values())
)

print("\nFirst 10 repeated raw pairs:")
for pair, count in list(raw_duplicate_pairs.items())[:10]:
    print(pair, "occurrences:", count)

# Check whether the normalized duplicates were already duplicated in raw data
raw_duplicate_count_for_normalized_examples = sum(
    1
    for pair in duplicate_final_pairs
    if raw_pair_counts.get(
        (pair[0], pair[1][:-1]), 0
    ) > 1
)

print(
    "\nNormalized duplicate pairs with a matching repeated raw pair:",
    raw_duplicate_count_for_normalized_examples
)


Raw enrolment records: 33997
Unique raw student-exam pairs: 33866
Distinct repeated raw pairs: 122
Extra raw occurrences: 131

First 10 repeated raw pairs:
('A940014523', 'C81J5AE') occurrences: 2
('A940022595', 'C81J3AE') occurrences: 2
('A940026445', 'C81J2AE') occurrences: 2
('A940031287', 'C81J2AE') occurrences: 2
('A940033292', 'C81J2AE') occurrences: 2
('A940034646', 'C81J2AE') occurrences: 2
('A940042722', 'C81J3AE') occurrences: 2
('A945073318', 'C81J4AE') occurrences: 2
('A945311483', 'C81J3AE') occurrences: 2
('A946008014', 'C81J5AE') occurrences: 2

Normalized duplicate pairs with a matching repeated raw pair: 122


In [9]:

# Create a clean set of unique student-exam relationships for analysis
unique_final_pairs = sorted(set(final_pairs))

# Final validation checks
assert len(student_id_set) == 7896
assert len(exam_id_set) == 800
assert len(final_pairs) == 33997
assert len(unmatched_final_ids) == 0
assert len(final_unknown_students) == 0
assert len(final_unknown_exams) == 0
assert len(unique_final_pairs) == 33866

validation_results = pd.DataFrame([
    {"Metric": "Unique students", "Value": len(student_id_set)},
    {"Metric": "Unique exams", "Value": len(exam_id_set)},
    {"Metric": "Raw enrolment records", "Value": len(student_exam_pairs)},
    {"Metric": "Mapped enrolment records", "Value": len(final_pairs)},
    {"Metric": "Unique student-exam pairs", "Value": len(unique_final_pairs)},
    {"Metric": "Distinct duplicated pairs", "Value": len(raw_duplicate_pairs)},
    {"Metric": "Extra duplicate occurrences", "Value": sum(c - 1 for c in raw_duplicate_pairs.values())},
    {"Metric": "Unmatched exam IDs", "Value": len(unmatched_final_ids)},
    {"Metric": "Unknown student references", "Value": len(final_unknown_students)},
    {"Metric": "Unknown exam references", "Value": len(final_unknown_exams)},
])

display(validation_results)

print("All required integrity checks passed.")
print("Original enrolment data has not been modified.")


,Metric,Value
0,Unique students,7896
1,Unique exams,800
2,Raw enrolment records,33997
3,Mapped enrolment records,33997
4,Unique student-exam pairs,33866
5,Distinct duplicated pairs,122
6,Extra duplicate occurrences,131
7,Unmatched exam IDs,0
8,Unknown student references,0
9,Unknown exam references,0


All required integrity checks passed.
Original enrolment data has not been modified.



## Validation Conclusions

The University of Nottingham benchmark contains 7,896 unique
students, 800 examinations, and 33,997 raw enrolment records.

The enrolment exam identifiers required normalization to match
the examination identifiers. One special mapping was applied:
`K1AHWAE` to `K1AHWAE2`.

All enrolment records were successfully mapped, with no unknown
student or examination references remaining.

The raw enrolment file contains 122 duplicated student–exam
pairs, accounting for 131 extra occurrences. These records are
documented rather than deleted from the original dataset.
For subsequent conflict-graph analysis, unique student–exam
relationships will be used.

The validated data is ready for conflict graph construction.
